# Отбор фичей

**Шаг 4 плана.** Удаление шумных фичей (низкая вариативность, много NaN, мультиколлинеарность), выбор лучших по AUC и importance для финального target `tp_sl_1_05`.

## 1. Импорты и загрузка

In [13]:
import sys
import os
import numpy as np
import pandas as pd

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

from src.features.feature_pipeline import get_feature_columns

TARGET_COL = 'target'
df = pd.read_parquet(os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet'))
corr_path = os.path.join(_root, 'outputs', 'correlation_with_target_tp_sl_1_05.csv')
if os.path.exists(corr_path):
    corr_df = pd.read_csv(corr_path)
    print('Загружена correlation_with_target_tp_sl_1_05.csv')
else:
    corr_df = None
    print('correlation_with_target_tp_sl_1_05.csv не найден — запустите 05')

feat_base = [c for c in get_feature_columns(include_symbol=True)]
feat_all = [c for c in feat_base if c in df.columns] + [c for c in ['rd_regime', 'rd_regime_transition'] if c in df.columns and c not in feat_base]
print(f'Всего фичей в датасете: {len(feat_all)}')

# Применяем ручные исключения из 05 (excluded_features_tp_sl_1_05.txt)
excluded_path = os.path.join(_root, 'outputs', 'excluded_features_tp_sl_1_05.txt')
if os.path.exists(excluded_path):
    with open(excluded_path, encoding='utf-8') as f:
        excluded = [l.strip() for l in f if l.strip()]
    feat_cols = [c for c in feat_all if c not in excluded]
    print(f'Применены ручные исключения из 05: {excluded}')
else:
    feat_cols = feat_all
    excluded = []
    print('excluded_features_tp_sl_1_05.txt не найден — используем все фичи')

print(f'После исключений: {len(feat_cols)} фичей')

Загружена correlation_with_target_tp_sl_1_05.csv
Всего фичей в датасете: 27
Применены ручные исключения из 05: ['is_weekend', 'bb_position', 'bb_width']
После исключений: 24 фичей


## 2. Удаление шумных фичей

In [14]:
valid = df.dropna(subset=[TARGET_COL] + feat_cols)
valid = valid[valid[TARGET_COL].isin([1.0, -1.0])]

# Низкая вариативность (std ≈ 0)
std_thresh = 1e-8
low_var = [c for c in feat_cols if valid[c].std() < std_thresh]

# Высокая доля NaN
nan_pct = df[feat_cols].isna().mean()
high_nan = [c for c in feat_cols if nan_pct[c] > 0.5]

noisy = list(set(low_var + high_nan))
feat_clean = [c for c in feat_cols if c not in noisy]
print('Удалены (шумные):', noisy if noisy else '(нет)')
print(f'После удаления: {len(feat_clean)} фичей')

Удалены (шумные): (нет)
После удаления: 24 фичей


## 3. Мультиколлинеарность (|r| > 0.85)

In [15]:
corr_m = valid[feat_clean].corr()
to_drop = set()
importance = corr_df.set_index('feature')['abs_corr'].to_dict() if corr_df is not None else {}

for i in range(len(corr_m.columns)):
    for j in range(i + 1, len(corr_m.columns)):
        a, b = corr_m.columns[i], corr_m.columns[j]
        if a in to_drop or b in to_drop:
            continue
        r = corr_m.iloc[i, j]
        if abs(r) > 0.85:
            imp_a = importance.get(a, 0)
            imp_b = importance.get(b, 0)
            drop = b if imp_a >= imp_b else a
            to_drop.add(drop)

feat_multi = [c for c in feat_clean if c not in to_drop]
print('Удалены (мультиколлинеарность):', list(to_drop) if to_drop else '(нет)')
print(f'После удаления: {len(feat_multi)} фичей')

Удалены (мультиколлинеарность): ['macd_line']
После удаления: 23 фичей


## 4. AUC и XGB importance

In [16]:
from sklearn.metrics import roc_auc_score

y = (valid[TARGET_COL] == 1).astype(int).values
aucs = []
for c in feat_multi:
    x = valid[c].fillna(valid[c].median()).values
    try:
        auc = roc_auc_score(y, x)
        auc = max(auc, 1 - auc)  # direction-free
    except Exception:
        auc = 0.5
    aucs.append({'feature': c, 'auc': auc})

auc_df = pd.DataFrame(aucs).sort_values('auc', ascending=False)
print('Топ-10 по AUC (direction-free):')
print(auc_df.head(10).to_string(index=False))

Топ-10 по AUC (direction-free):
        feature      auc
       rd_mom_5 0.566018
       rd_mom_1 0.563399
   rd_zscore_30 0.556617
      rd_mom_10 0.552870
      rd_regime 0.542286
rd_acceleration 0.525170
      rd_ema_20 0.513211
 close_position 0.509309
  volatility_14 0.507658
          ret_1 0.506987


In [17]:
try:
    import xgboost as xgb
    model = xgb.XGBClassifier(n_estimators=50, max_depth=4, random_state=42)
    X = valid[feat_multi].fillna(0)
    model.fit(X, y)
    imp = pd.DataFrame({'feature': feat_multi, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
    print('Топ-10 по XGB importance:')
    print(imp.head(10).to_string(index=False))
except ImportError:
    imp = None
    print('XGBoost не установлен — используем только AUC')

Топ-10 по XGB importance:
        feature  importance
       rd_mom_5    0.108111
       rd_mom_1    0.101474
          ret_5    0.091792
          ret_1    0.069564
   rd_zscore_30    0.055536
      macd_hist    0.050342
         rsi_14    0.049611
        dow_cos    0.046198
        dow_sin    0.042028
rd_acceleration    0.040669


## 4a. Leakage-check и Ablation

Проверка на утечку (ret_1, rd_regime, rd_mom_*: |corr| < 0.9) и быстрый ablation по группам фичей.

In [ ]:
# Leakage-check: ret_1, rd_regime, rd_mom_* не должны быть |corr| > 0.9
if corr_df is not None and 'abs_corr' in corr_df.columns:
    leak_corr = corr_df[corr_df['feature'].isin(feat_multi)][['feature', 'corr', 'abs_corr']]
else:
    leak_corr = pd.DataFrame([
        {'feature': c, 'corr': valid[c].corr(valid[TARGET_COL]), 'abs_corr': abs(valid[c].corr(valid[TARGET_COL]))}
        for c in feat_multi
    ])
leak_features = [c for c in ['ret_1', 'rd_regime'] + [f for f in feat_multi if f.startswith('rd_mom_')] if c in feat_multi]
leak_check = leak_corr[leak_corr['feature'].isin(leak_features)]
print('Leakage-check (ret_1, rd_regime, rd_mom_*):')
if len(leak_check) > 0:
    print(leak_check.to_string(index=False))
    if (leak_check['abs_corr'] > 0.9).any():
        print('\nВНИМАНИЕ: возможный leakage — |corr| > 0.9')
    else:
        print('\nПодозрительных фичей (|corr| > 0.9) не обнаружено.')
else:
    print('  (нет проверяемых фичей в feat_multi)')

In [ ]:
# Ablation: base_all, drop_time, drop_symbol, drop_rd (LogReg, session-aware)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

sessions = list(valid['session_key'].unique())
np.random.seed(42)
np.random.shuffle(sessions)
n_train = int(0.8 * len(sessions))
train_sessions = set(sessions[:n_train])
test_sessions = set(sessions[n_train:])
train = valid[valid['session_key'].isin(train_sessions)]
test = valid[valid['session_key'].isin(test_sessions)]
y_train = (train[TARGET_COL] == 1).astype(int).values
y_test = (test[TARGET_COL] == 1).astype(int).values

time_cols = [c for c in ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos'] if c in feat_multi]
feature_sets = {
    'base_all': feat_multi,
    'drop_time': [c for c in feat_multi if c not in time_cols],
    'drop_symbol': [c for c in feat_multi if c != 'symbol_encoded'],
    'drop_rd': [c for c in feat_multi if not (c.startswith('rd_') or c == 'abs_rd')],
}

rows = []
for name, cols in feature_sets.items():
    if len(cols) == 0:
        continue
    X_tr = train[cols].fillna(0).values
    X_te = test[cols].fillna(0).values
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr)
    X_te = sc.transform(X_te)
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_tr, y_train)
    auc = roc_auc_score(y_test, lr.predict_proba(X_te)[:, 1])
    rows.append({'set': name, 'n_features': len(cols), 'auc': auc})

ablation_df = pd.DataFrame(rows).sort_values('auc', ascending=False)
print('Ablation (LogReg, session-aware, StandardScaler):')
print(ablation_df.to_string(index=False))

### Исключение symbol_encoded (по результатам анализа)

По итогам AUC, XGB importance и ablation (см. выше):
- Корреляция с target: |r| = 0.003 — практически ноль.
- AUC одномерный: 0.005 — не лучше рандома.
- XGB importance: 0.31% — на уровне шума.
- Ablation: AUC без symbol (0.6151) >= AUC с symbol (0.6142).

Вывод: `symbol_encoded` не несёт полезной информации для модели, а в API потребует поддержки LabelEncoder. Исключаем.

In [18]:
drop_by_analysis = ['symbol_encoded']
feat_final = [c for c in feat_multi if c not in drop_by_analysis]
print(f'Исключены по результатам анализа: {drop_by_analysis}')
print(f'После исключения: {len(feat_final)} фичей')

Исключены по результатам анализа: ['symbol_encoded']
После исключения: 22 фичей


## 5. Финальный список FEATURES_SELECTED

In [19]:
# Финальный список: ручные исключения (05) + шум + мультиколлинеарность + анализ (06)
FEATURES_SELECTED = feat_final

os.makedirs(os.path.join(_root, 'outputs'), exist_ok=True)
out_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')
with open(out_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(FEATURES_SELECTED))

print(f'FEATURES_SELECTED: {len(FEATURES_SELECTED)} фичей')
print(FEATURES_SELECTED)
print()
if excluded:
    print(f'Ручные исключения (из 05): {excluded}')
if noisy:
    print(f'Шумные (авто): {noisy}')
if to_drop:
    print(f'Мультиколлинеарные (авто, |r|>0.85): {list(to_drop)}')
if drop_by_analysis:
    print(f'По результатам анализа (06): {drop_by_analysis}')
print(f'\nСохранено: {out_path}')
print('Этот файл — единственный источник правды для 07, обучения и API.')

FEATURES_SELECTED: 22 фичей
['rd_mom_1', 'rd_mom_5', 'rd_mom_10', 'rd_acceleration', 'rd_zscore_30', 'rd_ema_20', 'abs_rd', 'ret_1', 'ret_5', 'rsi_14', 'macd_signal', 'macd_hist', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'rd_regime', 'rd_regime_transition']

Ручные исключения (из 05): ['is_weekend', 'bb_position', 'bb_width']
Мультиколлинеарные (авто, |r|>0.85): ['macd_line']
По результатам анализа (06): ['symbol_encoded']

Сохранено: c:\project\trading_bot_2Engine\outputs\features_selected_tp_sl_1_05.txt
Этот файл — единственный источник правды для 07, обучения и API.


## Итог шага 4

- Применены ручные исключения из 05 (`excluded_features_tp_sl_1_05.txt`): `is_weekend`, `bb_position`, `bb_width`.
- Удалены шумные фичи (низкая вариативность, высокий % NaN).
- Удалены дубликаты по мультиколлинеарности (|r| > 0.85): `macd_line`.
- Исключен `symbol_encoded` — по результатам AUC, importance и ablation не несёт информации, упрощает API.
- Оставшиеся фичи оценены по AUC и XGB importance.
- **Единственный источник правды:** `outputs/features_selected_tp_sl_1_05.txt` — используется в 07, обучении и API.

### Про ликвидность и объём
- Сейчас из объёмных фичей есть `volume_rel_20` (относительный объём к MA20).
- Градация по ликвидности/капитализации **не добавляется** на текущем этапе: 214 символов в датасете, `symbol_encoded` не значим → модель уже работает универсально.
- Вернуться к вопросу при расширении пула (50+ инструментов) или деградации на отдельных монетах.